# Segmentation 参数调试 - HCC项目
围绕已知最优参数 (block_size=13, offset=0.03, min_distance=11, expand_distance=5) 做网格搜索

**测试样本**: Y10393D8（选一个代表即可，确认参数后批量跑其余8个）

In [ ]:
import sys
sys.path.append('/data/work/liver_hcc_spatial/xxx/Tools')
from Segmentation import Segmentation
import matplotlib.pyplot as plt
import numpy as np
import cv2
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ===== 配置 =====
SAMPLE_ID = 'Y10393D8'

# ssDNA 配准后的 PNG 路径（PS registration 输出）
IMG_PATH = f'/data/work/liver_hcc_spatial/00_raw/{SAMPLE_ID}.mRNA(match) 拷贝.png'



# 参数测试结果保存目录
OUT_DIR = f'/data/work/liver_hcc_spatial/03_segmentation/param_test/{SAMPLE_ID}'
os.makedirs(OUT_DIR, exist_ok=True)

print(f'输出目录: {OUT_DIR}')
print(f'图像路径: {IMG_PATH}')
print(f'图像是否存在: {os.path.exists(IMG_PATH)}')

In [ ]:
# 加载图像
sobj = Segmentation()
sobj.load(
    img_path=IMG_PATH,
    mRNA_path=None,
)

# ★ 立即裁剪到中心 2000×2000，后续所有操作在小图上进行（避免全图 OOM）
h, w = sobj.raw_img.shape[:2]
row_start = h // 2 - 1000
row_end   = h // 2 + 1000
col_start = w // 2 - 1000
col_end   = w // 2 + 1000
sobj.raw_img = sobj.raw_img[row_start:row_end, col_start:col_end].copy()
print(f'裁剪后图像尺寸: {sobj.raw_img.shape}')

In [ ]:
# ===== 裁剪区域预览 =====
# raw_img 已在上一步裁好，直接展示
print(f'图像尺寸: {sobj.raw_img.shape}')

plt.figure(figsize=(6, 6))
plt.imshow(sobj.raw_img, cmap='gray')
plt.title('裁剪测试区域预览（中心 2000×2000）')
plt.axis('off')
plt.show()

## 1. 网格搜索：block_size × min_distance
固定 offset=0.03, expand_distance=5，扫描 block_size 和 min_distance

In [ ]:
# 参数范围（以已知最优为中心上下浮动）
block_sizes    = [9, 11, 13, 15, 17]   # 已知最优: 13
min_distances  = [7, 9, 11, 13, 15]    # 已知最优: 11
offset         = 0.03                   # 固定
expand_dist    = 5                      # 固定

print(f'共 {len(block_sizes) * len(min_distances)} 组参数')
print(f'block_size: {block_sizes}')
print(f'min_distance: {min_distances}')

In [ ]:
print([m for m in dir(sobj) if not m.startswith('_')])
sobj.pre_process()


In [ ]:
for bs in block_sizes:
    for md in min_distances:
        sobj.watershed(
            block_size=bs,
            offset=offset,
            min_distance=md,
            expand_distance=expand_dist,
            verbose=False
        )
        # raw_img 已是 2000×2000，sobj.label 就是裁剪区域，直接保存
        out_path = os.path.join(OUT_DIR, f'bs{bs}_md{md}_ot{offset}_ed{expand_dist}.png')
        cv2.imwrite(out_path, sobj.label)
        print(f'已保存: bs={bs}, md={md} -> {os.path.basename(out_path)}')

print('\n网格搜索完成！')

## 2. 可视化对比（在notebook内直接看）

In [ ]:
fig, axes = plt.subplots(len(block_sizes), len(min_distances),
                          figsize=(4 * len(min_distances), 4 * len(block_sizes)))

for i, bs in enumerate(block_sizes):
    for j, md in enumerate(min_distances):
        img_path = os.path.join(OUT_DIR, f'bs{bs}_md{md}_ot{offset}_ed{expand_dist}.png')
        img = cv2.imread(img_path)
        if img is not None:
            axes[i, j].imshow(img)
        axes[i, j].set_title(f'bs={bs}, md={md}', fontsize=9)
        axes[i, j].axis('off')

plt.suptitle(f'{SAMPLE_ID} | offset={offset}, expand_dist={expand_dist}', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'grid_overview.png'), dpi=100, bbox_inches='tight')
plt.show()
print('汇总图已保存: grid_overview.png')

## 3. （可选）固定 bs/md，扫描 expand_distance
在确定 block_size 和 min_distance 之后运行

In [ ]:
# 填入上面选出的最优参数
BEST_BS = 15
BEST_MD = 7
BEST_OT = 0.03

expand_distances = [2, 3, 4, 5, 6, 7, 8]

fig, axes = plt.subplots(1, len(expand_distances),
                          figsize=(4 * len(expand_distances), 5))

for j, ed in enumerate(expand_distances):
    sobj.watershed(
        block_size=BEST_BS,
        offset=BEST_OT,
        min_distance=BEST_MD,
        expand_distance=ed,
        verbose=False
    )
    # sobj.label 已是裁剪尺寸，直接显示
    axes[j].imshow(sobj.label)
    axes[j].set_title(f'ed={ed}', fontsize=10)
    axes[j].axis('off')

plt.suptitle(f'expand_distance 扫描 | bs={BEST_BS}, md={BEST_MD}, ot={BEST_OT}', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'expand_dist_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ===== 最终确认：用选好的参数跑一次完整预览 =====
FINAL_BS = 13
FINAL_MD = 11
FINAL_OT = 0.03
FINAL_ED = 5

sobj.watershed(
    block_size=FINAL_BS,
    offset=FINAL_OT,
    min_distance=FINAL_MD,
    expand_distance=FINAL_ED,
    verbose=True
)

plt.figure(figsize=(10, 10))
# sobj.label 已是裁剪尺寸，直接显示
plt.imshow(sobj.label)
plt.title(f'最终参数: bs={FINAL_BS}, md={FINAL_MD}, ot={FINAL_OT}, ed={FINAL_ED}')
plt.axis('off')
plt.show()